<a href="https://colab.research.google.com/github/kayeneii/Floodgate/blob/main/floodgate_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mapping Flood Risk Zones Using Open Data

## *Step 1: Data Collection*
- Choose study area and export small rasters from GEE to test.

- Define a bounding box (min lon, min lat, max lon, max lat).

- Use this to preview DEM, rivers, and rainfall quickly.

Open: https://code.earthengine.google.com/cd290a1d5be41113879d68813786bc9a

## *Step 2: Preprocessing*

- Authenticate: Earth Engine & Google Drive.

- Define study area and date window.

- Query CHIRPS and check collection size — stop with a clear message if 0.

- Aggregate (sum) and reproject to DEM projection.

- Export images to Drive using ee.batch.Export.image.toDrive() and start the tasks

- Poll / print task.status() to know how it started.

In [59]:
# Colab setup: install required Python packages
!pip install earthengine-api geemap rioxarray xarray rasterio geopandas osmnx fiona folium contextily shapely pyproj -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 58.3 MB/s eta 0:00:00


In [36]:
# Authenticate EE and mount Google Drive
import ee, time, os
ee.Authenticate()   # follow interactive prompt
ee.Initialize(project="thinking-league-472119-r1")

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
# PARAMETERS - edit these
minLon, minLat, maxLon, maxLat = 3.0, 6.0, 4.0, 7.0
start_date = '2025-04-01'
end_date   = '2025-06-30'
export_scale = 30  # meters
drive_folder = 'GEE_exports'  # will be created in your Drive root

In [49]:
# Build study geometry
study = ee.Geometry.Rectangle([minLon, minLat, maxLon, maxLat])

In [52]:
# Build CHIRPS collection and safe-check size
chirps_col = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate(start_date, end_date).filterBounds(study)

In [53]:
# IMPORTANT: get size server-side and then client-side
chirps_count = chirps_col.size().getInfo()  # safe in Colab, but will block briefly
print('CHIRPS count for your filters =', chirps_count)
if chirps_count == 0:
    raise SystemExit('ERROR: No CHIRPS images found for that date range and region. Try different dates (CHIRPS available from 1981-01-01) or expand the region.')

CHIRPS count for your filters = 90


In [54]:
# Aggregate to seasonal total
chirps_total = chirps_col.sum().clip(study).toFloat()

In [55]:
# DEM
dem = ee.Image('USGS/SRTMGL1_003').clip(study).toFloat()
slope = ee.Terrain.slope(dem)

In [56]:
# Ensure rainfall image projection matches DEM (reproject to DEM's projection)
chirps_total = chirps_total.reproject(crs=dem.projection(), scale=export_scale)

In [57]:
# === Export functions with robust start + status printing ===
def export_image_to_drive(image, description, file_name_prefix, folder, region, scale, maxPixels=1e10):
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=file_name_prefix,
        region=region,
        scale=scale,
        maxPixels=maxPixels
    )
    task.start()
    print(f'Started task {description} -> {folder}/{file_name_prefix}. Monitoring status...')
    # Print status for a few seconds
    for i in range(30):  # polls for ~30*2s = 60s
        status = task.status()
        print(i, status['state'])
        if status['state'] in ('COMPLETED', 'FAILED', 'CANCELLED'):
            break
        time.sleep(2)
    print('Final status:', task.status())
    return task

In [66]:
# Start exports (small bbox for testing)
t1 = export_image_to_drive(chirps_total, 'floodgate_chirps_seasonal', 'floodgate_chirps_seasonal', drive_folder, study, export_scale)
t2 = export_image_to_drive(dem, 'floodgate_dem_sample', 'floodgate_dem_sample', drive_folder, study, export_scale)
t3 = export_image_to_drive(slope, 'floodgate_slope', 'floodgate_slope_sample', drive_folder, study, export_scale)

print('Export tasks submitted. Open https://drive.google.com/drive/my-drive and look for the folder:', drive_folder)

Started task floodgate_chirps_seasonal -> GEE_exports/floodgate_chirps_seasonal. Monitoring status...
0 READY
1 READY
2 READY
3 READY
4 READY
5 READY
6 RUNNING
7 RUNNING
8 RUNNING
9 RUNNING
10 RUNNING
11 RUNNING
12 RUNNING
13 RUNNING
14 RUNNING
15 RUNNING
16 RUNNING
17 RUNNING
18 RUNNING
19 RUNNING
20 RUNNING
21 RUNNING
22 RUNNING
23 RUNNING
24 RUNNING
25 RUNNING
26 RUNNING
27 RUNNING
28 RUNNING
29 RUNNING
Final status: {'state': 'RUNNING', 'description': 'floodgate_chirps_seasonal', 'priority': 100, 'creation_timestamp_ms': 1760993978493, 'update_timestamp_ms': 1760994030447, 'start_timestamp_ms': 1760993989482, 'task_type': 'EXPORT_IMAGE', 'attempt': 1, 'batch_eecu_usage_seconds': 40.267, 'id': 'JJIJQHWFAVVVFQMDPUPOIBDA', 'name': 'projects/thinking-league-472119-r1/operations/JJIJQHWFAVVVFQMDPUPOIBDA'}
Started task floodgate_dem_sample -> GEE_exports/floodgate_dem_sample. Monitoring status...
0 READY
1 READY
2 READY
3 READY
4 RUNNING
5 RUNNING
6 RUNNING
7 RUNNING
8 RUNNING
9 RUNNING


## *Step 3: Flood Risk Index (FRI) Construction*

In [67]:
# Load weights
weights = {
 "w1": 0.5,
 "w2": 0.35,
 "w3": 0.15,
 "slope_threshold": 5,
 "river_buffer_m": 1000,
 "fri_threshold": 0.6
}

In [69]:
# Paths to your exported tif files from Drive (mount Drive in Colab, or upload)
# Assuming the files are in your Google Drive in the 'GEE_exports' folder
chirps_fp = '/content/drive/MyDrive/GEE_exports/floodgate_chirps_seasonal.tif'
dem_fp = '/content/drive/MyDrive/GEE_exports/floodgate_dem_sample.tif'
slope_fp = '/content/drive/MyDrive/GEE_exports/floodgate_slope_sample.tif'

In [70]:
# Load rasters with rioxarray
import rioxarray as rxr
chirps = rxr.open_rasterio(chirps_fp, masked=True).squeeze()
dem = rxr.open_rasterio(dem_fp, masked=True).squeeze()
slope = rxr.open_rasterio(slope_fp, masked=True).squeeze()

In [72]:
# Align/resample to same grid if necessary
import xarray as xr
chirps, dem = xr.align(chirps, dem, join='override')
slope = slope.rio.reproject_match(dem)

In [73]:
# Normalize helper
def normalize(arr):
    a = arr.values.astype('float32')
    mask = np.isfinite(a)
    mini, maxi = np.nanmin(a[mask]), np.nanmax(a[mask])
    out = np.full(a.shape, np.nan, dtype='float32')
    if maxi > mini:
        out[mask] = (a[mask] - mini) / (maxi - mini)
    else:
        out[mask] = 0.0
    return xr.DataArray(out, coords=arr.coords, dims=arr.dims)

In [75]:
import numpy as np
norm_rain = normalize(chirps)
norm_elev = normalize(dem)  # higher elevation -> lower risk in our formula
norm_slope = normalize(slope)

In [76]:
# Slope contribution: give slope only above threshold (optional)
slope_mask = (slope >= weights['slope_threshold']).astype('float32')
norm_slope_adj = norm_slope * slope_mask

In [77]:
# FRI = w1*rain + w2*(1 - elev) + w3*slope_adj
fri = (weights['w1'] * norm_rain +
       weights['w2'] * (1.0 - norm_elev) +
       weights['w3'] * norm_slope_adj)

fri = fri.clip(min=0, max=1)

In [79]:
# Save FRI GeoTIFF
fri.rio.write_crs(dem.rio.crs)
out_fp = '/content/drive/MyDrive/floodgate_fri_geotiff.tif'
fri.rio.to_raster(out_fp)
print('Saved FRI to', out_fp)

Saved FRI to /content/floodgate_fri_geotiff.tif


Notes on weights and normalization

- Default values are a starting point. Keep weights in a JSON so analysts can run sensitivity tests.

- Normalization uses min/max across the study area; for larger regions consider robust scaling (percentiles) to avoid outliers dominating.